# **Accessibilty Infrastructure - NYC Subway Stations**

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import plotly.express as px
import requests

url_stations = "https://data.ny.gov/api/views/39hk-dx4f/rows.csv?accessType=DOWNLOAD"
stations = pd.read_csv(url_stations)
print(stations.shape)
stations.head()

## **Initial Analysis (Subway Accessibility)**

In [ ]:
print(stations["ADA"].value_counts())
print("\n")
print(stations["ADA"].value_counts(normalize=True).round(2) * 100)

ada_labels = {
    0: "Not accessible",
    1: "Fully accessible",
    2: "Partially accessible"
}

stations["ADA_Status"] = stations["ADA"].map(ada_labels)
stations[["ADA", "ADA_Status"]].head(10)

ada_colors = {
    "Fully accessible": "green",
    "Partially accessible": "yellow",
    "Not accessible": "red"
}

stations["color"] = stations["ADA_Status"].map(ada_colors)
stations[["Stop Name", "ADA_Status", "color"]].head(10)

In [ ]:
url_equipement = "https://data.ny.gov/api/views/94fv-bak7/rows.csv?accessType=DOWNLOAD"
equipment = pd.read_csv(url_equipement)
print(equipment.shape)
equipment.head()

print(equipment["Elevator or Escalator"].value_counts())

print("Unique stations with equipment:", equipment["Station Name"].nunique())
print("Total stations:", stations["Stop Name"].nunique())

In [ ]:
mrn_values = set(equipment["Station MRN"].dropna().astype(int))
station_id_values = set(stations["Station ID"])
overlap = mrn_values.intersection(station_id_values)
print(f"Number of matching IDs: {len(overlap)}")
print(f"Sample matches: {list(overlap)[:10]}")

elevators_per_station = equipment[equipment["Elevator or Escalator"] == "Elevator"].groupby("Station MRN").size().reset_index()
elevators_per_station.columns = ["Station ID", "elevator_count"]
elevators_per_station["Station ID"] = elevators_per_station["Station ID"].astype(int)
print(elevators_per_station.head(10))

### **Join Data**

In [ ]:
stations["Station ID"] = stations["Station ID"].astype(int)

stations = stations.merge(elevators_per_station, on="Station ID", how="left")

stations["elevator_count"] = stations["elevator_count"].fillna(0).astype(int)

print(stations[["Stop Name", "ADA_Status", "elevator_count"]].head(10))

### **Live Status**

In [ ]:
url_availability = "https://data.ny.gov/api/views/rc78-7x78/rows.csv?accessType=DOWNLOAD"
availability = pd.read_csv(url_availability)
print(availability.shape)
availability.head()

elevator_reliability = availability[availability["Equipment Type"] == "Elevator"].groupby("Equipment Code")["24-Hour Availability"].mean().reset_index()
elevator_reliability.columns = ["Equipment Code", "avg_availability"]
elevator_reliability["avg_availability"] = elevator_reliability["avg_availability"].round(3)
print(elevator_reliability.sort_values("avg_availability").head(10))

station_reliability = availability[availability["Equipment Type"] == "Elevator"].groupby("Station MRN")["24-Hour Availability"].mean().reset_index()
station_reliability.columns = ["Station ID", "avg_elevator_availability"]
station_reliability["avg_elevator_availability"] = station_reliability["avg_elevator_availability"].round(3)
station_reliability["Station ID"] = station_reliability["Station ID"].astype(int)
print(station_reliability.sort_values("avg_elevator_availability").head(10))

worst_stations = station_reliability.sort_values("avg_elevator_availability").head(10)
worst_stations = worst_stations.merge(
    stations[["Station ID", "Stop Name", "Borough", "ADA_Status"]],
    on="Station ID",
    how="left"
)
print(worst_stations[["Stop Name", "Borough", "ADA_Status", "avg_elevator_availability"]])

In [ ]:
stations = stations.merge(
    station_reliability,
    on="Station ID",
    how="left"
)

stations["avg_elevator_availability"] = stations["avg_elevator_availability"].fillna(0)

print(stations.shape)
print(stations[["Stop Name", "ADA_Status", "elevator_count", "avg_elevator_availability"]].head(10))

In [ ]:
borough_summary = stations.groupby("Borough").agg(
    total_stations=("Stop Name", "count"),
    fully_accessible=("ADA", lambda x: (x == 1).sum()),
    not_accessible=("ADA", lambda x: (x == 0).sum()),
    avg_availability=("avg_elevator_availability", "mean")
).reset_index()

borough_summary["pct_accessible"] = (borough_summary["fully_accessible"] / borough_summary["total_stations"] * 100).round(1)

print(borough_summary)

borough_names = {
    "Bk": "Brooklyn",
    "Bx": "Bronx",
    "M": "Manhattan",
    "Q": "Queens",
    "SI": "Staten Island"
}

borough_summary["Borough"] = borough_summary["Borough"].map(borough_names)
stations["Borough_Full"] = stations["Borough"].map(borough_names)

print(borough_summary)

citywide_pct = round(stations[stations["ADA"] == 1].shape[0] / stations.shape[0] * 100, 1)
print(f"Citywide accessible stations: {citywide_pct}%")

## **Visualization**

In [ ]:
fig = px.bar(
    borough_summary,
    x="Borough",
    y="pct_accessible",
    color="pct_accessible",
    color_continuous_scale=["#F4C0D1", "#D4537E", "#72243E"],
    title=f"Percentage of fully accessible subway stations by borough — citywide average: {citywide_pct}%",
    labels={"Borough": "Borough", "pct_accessible": "Accessible stations (%)"}
)

fig.add_hline(
    y=citywide_pct,
    line_dash="dash",
    line_color="black",
    annotation_font_color="red",
    annotation_text=f"NYC average: {citywide_pct}%",
    annotation_position="top right"
)

fig.update_traces(hovertemplate="<b>%{x}</b><br>Accessible: %{y}%<extra></extra>")
fig.update_layout(xaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
url_lines = "https://data.ny.gov/resource/s692-irgq.geojson?$limit=500"
lines = gpd.read_file(url_lines)
print(lines.shape)
print(lines.columns.tolist())

line_colors = {
    "A": "#0039a6", "C": "#0039a6", "E": "#0039a6",
    "B": "#ff6319", "D": "#ff6319", "F": "#ff6319", "M": "#ff6319",
    "G": "#6cbe45",
    "L": "#a7a9ac",
    "J": "#996633", "Z": "#996633",
    "N": "#fccc0a", "Q": "#fccc0a", "R": "#fccc0a", "W": "#fccc0a",
    "1": "#ee352e", "2": "#ee352e", "3": "#ee352e",
    "4": "#00933c", "5": "#00933c", "6": "#00933c",
    "5 Peak": "#00933c",
    "7": "#b933ad",
    "T": "#00add0",
    "S": "#808183", "SF": "#808183", "ST": "#808183", "SR": "#808183",
    "SIR": "#87CEEB"
}

In [ ]:
m = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles="CartoDB positron",
    min_zoom=10,
    max_bounds=True
)

# draw subway lines first so station dots appear on top
for _, row in lines.iterrows():
    service = row["service"]
    color = line_colors.get(service, "#808183")
    folium.GeoJson(
        row["geometry"].__geo_interface__,
        style_function=lambda x, c=color: {
            "color": c,
            "weight": 3,
            "opacity": 0.8
        }
    ).add_to(m)

# draw station dots on top of lines
for _, row in stations.iterrows():
    folium.CircleMarker(
        location=[row["GTFS Latitude"], row["GTFS Longitude"]],
        radius=3,
        color=row["color"],
        fill=True,
        fill_color=row["color"],
        fill_opacity=0.8,
        tooltip=folium.Tooltip(
          f"<b>{row['Stop Name']}</b><br>"
          f"Lines: {row['Daytime Routes']}<br>"
          f"Borough: {row['Borough_Full']}<br>"
          f"Status: {row['ADA_Status']}<br>"
          f"Elevators: {row['elevator_count']}<br>"
          f"Elevator availability: {round(row['avg_elevator_availability'] * 100, 1)}%"
        )
    ).add_to(m)

m.fit_bounds([[40.4774, -74.2591], [40.9176, -73.7004]])
m